# Raw Data Inspection and Initial Workload Logic
This notebook documents the first inspection of the three raw datasets used in
the project:

 - European Football Injuries
 - Multimodal Sports Injury Dataset
 - University Football Injury Prediction Dataset

The goals of this notebook are:

1. Load and inspect the raw data
2. Compare structure, variables, and missing values
3. Assess join feasibility across the datasets
4. Compute an initial session-based workload metric from the Multimodal dataset
5. Derive rolling acute load, chronic load, and IRS
6. Prepare a first export for the training-session fact table

## 1. Load libraries and raw datasets
In this section, we import the required Python libraries and load the three
CSV files into pandas DataFrames.

In [10]:
import pandas as pd
import numpy as np

# Load raw datasets
european = pd.read_csv("full_dataset_thesis - 1.csv")
multimodal = pd.read_csv("multimodal_sports_injury_dataset.csv")
university = pd.read_csv("data.csv")

datasets = {
    "european": european,
    "multimodal": multimodal,
    "university": university
}

## 2. General dataset overview
To understand the structure of each source, we define a reusable function that
prints:
- shape
- column names
- data types
- missing values
- sample rows
- numeric summary
- most frequent categorical values

In [12]:
def dataset_overview(df, name):
    print("=" * 80)
    print(f"DATASET: {name.upper()}")
    print("=" * 80)

    print(f"Shape: {df.shape}")

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nDtypes:")
    print(df.dtypes)

    print("\nMissing values:")
    missing = pd.DataFrame({
        "missing_n": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }).sort_values("missing_pct", ascending=False)
    print(missing)

    print("\nSample rows:")
    print(df.head(3))

    print("\nNumeric summary:")
    print(df.describe(include=[np.number]).T)

    print("\nCategorical summary:")
    cat_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(cat_cols) > 0:
        for col in cat_cols:
            print(f"\nTop values in {col}:")
            print(df[col].value_counts(dropna=False).head(10))
    else:
        print("No categorical columns.")

In [13]:
for name, df in datasets.items():
    dataset_overview(df, name)

DATASET: EUROPEAN
Shape: (15603, 11)

Columns:
['Season', 'Injury', 'Days', 'Games missed', 'injury_from_parsed', 'injury_until_parsed', 'player_name', 'player_age', 'player_position', 'club', 'league']

Dtypes:
Season                   str
Injury                   str
Days                     str
Games missed           int64
injury_from_parsed       str
injury_until_parsed      str
player_name              str
player_age             int64
player_position          str
club                     str
league                   str
dtype: object

Missing values:
                     missing_n  missing_pct
Season                       0          0.0
Injury                       0          0.0
Days                         0          0.0
Games missed                 0          0.0
injury_from_parsed           0          0.0
injury_until_parsed          0          0.0
player_name                  0          0.0
player_age                   0          0.0
player_position              0          0.

## 3. Assess join feasibility across datasets
The next step is to check whether the three datasets can be linked directly at
player level.

We inspect which identity-related fields are actually available in each source.
This is important because the project initially assumed a more direct
player-level integration than the raw CSV files really allow.

In [14]:
print("\n" + "=" * 80)
print("JOIN FEASIBILITY CHECK")
print("=" * 80)

print("\nEuropean columns relevant for joins:")
print([
    c for c in european.columns
    if "player" in c.lower() or "club" in c.lower() or "date" in c.lower() or "position" in c.lower()
])

print("\nMultimodal columns relevant for joins:")
print([
    c for c in multimodal.columns
    if "athlete" in c.lower() or "session" in c.lower() or "date" in c.lower() or "sport" in c.lower()
])

print("\nUniversity columns relevant for joins:")
print([
    c for c in university.columns
    if
    "player" in c.lower() or "team" in c.lower() or "date" in c.lower() or "position" in c.lower() or "age" in c.lower()
])

print("\nCan University join to European on player name?")
print("player_name" in university.columns and "player_name" in european.columns)

print("\nCan Multimodal join to European on player name?")
print("player_name" in multimodal.columns and "player_name" in european.columns)

print("\nDoes Multimodal contain a real date column?")
print(any("date" in c.lower() for c in multimodal.columns))


JOIN FEASIBILITY CHECK

European columns relevant for joins:
['player_name', 'player_age', 'player_position', 'club']

Multimodal columns relevant for joins:
['athlete_id', 'session_id', 'sport_type']

University columns relevant for joins:
['Age', 'Position']

Can University join to European on player name?
False

Can Multimodal join to European on player name?
False

Does Multimodal contain a real date column?
False


**Interpretation of join feasibility**

The feasibility check shows that a real player-level merge across all three
datasets is not possible:

- The European dataset contains football player identity fields such as
   `player_name`, `club`, and real injury dates.
- The Multimodal dataset contains only an anonymous `athlete_id` and ordered
   `session_id`, but no player names and no real dates.
- The University dataset contains static player profiles, but no player names,
   no team names, and no shared identifier.

 Therefore, any cross-source integration must be handled at a more aggregated
 benchmark level rather than through a real shared player entity.

# 4. Calculate session-based load, acute load, chronic load, and IRS
Because the Multimodal dataset does not contain explicit calendar dates,
workload is operationalized using the order of sessions within each athlete.

In this first prototype:

- `session_load` = `training_load`
- `acute_load_7` = rolling 7-session average
- `chronic_load_28` = rolling 28-session average
- `irs` = `acute_load_7 / chronic_load_28`

This means the measure is currently **session-based**, not strictly
calendar-day-based.

In [15]:
multi = multimodal.copy()

# Sort sessions within each athlete by session order
multi = multi.sort_values(["athlete_id", "session_id"]).reset_index(drop=True)

# Use the provided training_load as session load
multi["session_load"] = multi["training_load"]

# Optional transparent comparison metric
multi["session_load_rpe_x_duration"] = (
        multi["training_intensity"] * multi["training_duration"]
)

# Rolling acute load: average of last 7 sessions
multi["acute_load_7"] = (
    multi.groupby("athlete_id")["session_load"]
    .transform(lambda s: s.rolling(window=7, min_periods=7).mean())
)

# Rolling chronic load: average of last 28 sessions
multi["chronic_load_28"] = (
    multi.groupby("athlete_id")["session_load"]
    .transform(lambda s: s.rolling(window=28, min_periods=28).mean())
)

# Injury Risk Score
multi["irs"] = multi["acute_load_7"] / multi["chronic_load_28"]

# Risk band classification
multi["risk_band"] = np.select(
    [
        multi["irs"] >= 1.5,
        (multi["irs"] >= 0.8) & (multi["irs"] < 1.5),
        multi["irs"] < 0.8
    ],
    [
        "high",
        "optimal",
        "under"
    ],
    default="insufficient_history"
)

In [16]:
print(
    multi[
        [
            "athlete_id",
            "session_id",
            "session_load",
            "acute_load_7",
            "chronic_load_28",
            "irs",
            "risk_band"
        ]
    ].head(35)
)

    athlete_id  session_id  session_load  acute_load_7  chronic_load_28  \
0            1           1    548.417962           NaN              NaN   
1            1           2    538.043815           NaN              NaN   
2            1           3    647.784339           NaN              NaN   
3            1           4    336.553431           NaN              NaN   
4            1           5    496.076352           NaN              NaN   
5            1           6    684.401046           NaN              NaN   
6            1           7   1081.065295    618.906034              NaN   
7            1           8    470.069442    607.713389              NaN   
8            1           9    826.062286    648.858884              NaN   
9            1          10    357.580668    607.401217              NaN   
10           1          11    714.948683    661.457682              NaN   
11           1          12    909.847637    720.567865              NaN   
12           1          1

**Interpretation of workload logic**

The first rows for each athlete are labeled `insufficient_history`, because a
28-session chronic workload cannot be computed immediately.

This is expected and methodologically correct:

- acute load requires at least 7 sessions
- chronic load requires at least 28 sessions
- IRS can only be calculated once both rolling measures are available

# 5. Summarize IRS distribution
To understand whether the calculated workload ratio behaves plausibly, we inspect:

- the distribution of risk bands
- the relative frequency of each band
- summary statistics of the IRS

In [17]:
print(multi["risk_band"].value_counts(dropna=False))
print((multi["risk_band"].value_counts(normalize=True, dropna=False) * 100).round(2))

risk_band
optimal                 9690
insufficient_history    4212
under                   1452
high                      66
Name: count, dtype: int64
risk_band
optimal                 62.84
insufficient_history    27.32
under                    9.42
high                     0.43
Name: proportion, dtype: float64


In [18]:
print(multi["irs"].describe())

count    11208.000000
mean         1.000367
std          0.179984
min          0.423281
25%          0.873852
50%          0.993192
75%          1.118508
max          1.702870
Name: irs, dtype: float64


**Interpretation of IRS distribution**

In the current prototype, most observations fall into the `optimal` band,
while a smaller share is classified as `under` and only a few observations are
classified as `high`.

This pattern is plausible because acute and chronic load are derived from the
same workload process, so the ratio is expected to cluster around 1 unless
there are stronger spikes or drops in session load.

# 6. Create a rounded result preview

For easier inspection and notebook readability, we create a rounded preview of
the key workload metrics.

In [19]:
result = multi[
    [
        "athlete_id",
        "session_id",
        "session_load",
        "acute_load_7",
        "chronic_load_28",
        "irs",
        "risk_band"
    ]
].copy()

result[["session_load", "acute_load_7", "chronic_load_28", "irs"]] = result[
    ["session_load", "acute_load_7", "chronic_load_28", "irs"]
].round(3)

print(result.head(35))

    athlete_id  session_id  session_load  acute_load_7  chronic_load_28  \
0            1           1       548.418           NaN              NaN   
1            1           2       538.044           NaN              NaN   
2            1           3       647.784           NaN              NaN   
3            1           4       336.553           NaN              NaN   
4            1           5       496.076           NaN              NaN   
5            1           6       684.401           NaN              NaN   
6            1           7      1081.065       618.906              NaN   
7            1           8       470.069       607.713              NaN   
8            1           9       826.062       648.859              NaN   
9            1          10       357.581       607.401              NaN   
10           1          11       714.949       661.458              NaN   
11           1          12       909.848       720.568              NaN   
12           1          1

# 7. Prepare first export for the training-session fact table

Based on the Multimodal dataset and the derived IRS logic, we create a first
structured export that can later be used as the basis for a
`fact_training_session` table in the database.

The export keeps:

- source identifiers
- workload measures
- derived IRS measures
- selected contextual athlete variables

In [20]:
fact_training_session_ready = multi[
    [
        "athlete_id",
        "session_id",
        "session_load",
        "acute_load_7",
        "chronic_load_28",
        "irs",
        "risk_band",
        "training_intensity",
        "training_duration",
        "fatigue_index",
        "injury_occurred",
        "sport_type",
        "gender",
        "age",
        "bmi"
    ]
].copy()

fact_training_session_ready = fact_training_session_ready.round({
    "session_load": 3,
    "acute_load_7": 3,
    "chronic_load_28": 3,
    "irs": 3,
    "training_intensity": 3,
    "training_duration": 3,
    "fatigue_index": 3,
    "bmi": 3
})

print(fact_training_session_ready.head(10))
print(fact_training_session_ready.shape)

   athlete_id  session_id  session_load  acute_load_7  chronic_load_28  irs  \
0           1           1       548.418           NaN              NaN  NaN   
1           1           2       538.044           NaN              NaN  NaN   
2           1           3       647.784           NaN              NaN  NaN   
3           1           4       336.553           NaN              NaN  NaN   
4           1           5       496.076           NaN              NaN  NaN   
5           1           6       684.401           NaN              NaN  NaN   
6           1           7      1081.065       618.906              NaN  NaN   
7           1           8       470.069       607.713              NaN  NaN   
8           1           9       826.062       648.859              NaN  NaN   
9           1          10       357.581       607.401              NaN  NaN   

              risk_band  training_intensity  training_duration  fatigue_index  \
0  insufficient_history               5.678      

**Summary**

This notebook showed that the three raw datasets differ strongly in
granularity and identity structure. A real player-level merge across all three
sources is not possible with the available fields. However, the Multimodal
dataset provides a usable basis for session-level workload analysis.

Using `training_load` as session load, we derived:

- rolling 7-session acute load
- rolling 28-session chronic load
- an Injury Risk Score (IRS)
- a first workload band classification

The resulting export provides a useful starting point for the database fact
table and for later SQL-based analytics in the project.

Because the Multimodal dataset does not contain explicit calendar dates, acute and chronic workload were operationalized as rolling 7-session and 28-session averages based on session order within each athlete.

The Multimodal dataset provides a direct workload variable (training_load), which we use as session load. Acute and chronic workload are calculated as rolling 7-session and 28-session averages per athlete, since the dataset does not contain explicit calendar dates. The Injury Risk Score (IRS) is then defined as the ratio of acute to chronic workload.

Based on the computed IRS, observations are classified into three bands: high (IRS ≥ 1.5), optimal (0.8 ≤ IRS < 1.5), and under (IRS < 0.8). Observations without sufficient history for a 28-session rolling window are labeled separately.